# 1. EDA Report — Bank Marketing Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
df = pd.read_csv('bank_marketing_updated_v1.csv')
print("Dataset loaded successfully")
print(f"Shape: {df.shape}")

## 2. Data Overview

In [ ]:
print("=== First 5 Rows ===")
display(df.head())
print("\n=== Last 5 Rows ===")
display(df.tail())
print("\n=== Data Types ===")
display(df.dtypes)
print("\n=== Dataset Info ===")
df.info()

## 3. Missing Value Analysis

In [ ]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['Missing Count'] > 0]
print("=== Missing Values ===")
display(missing)

if not missing.empty:
    plt.figure(figsize=(10, 6))
    sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
    plt.title('Missing Values Heatmap')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found in the dataset")

## 4. Statistical Summary

In [ ]:
print("=== Numerical Columns Summary ===")
display(df.describe())
print("\n=== Categorical Columns Summary ===")
display(df.describe(include='object'))

## 5. Distribution Analysis

In [ ]:
numerical_cols = df.select_dtypes(include=['int64','float64']).columns
if len(numerical_cols) > 0:
    df[numerical_cols].hist(bins=30, figsize=(15, 10))
    plt.suptitle('Numerical Columns Distribution')
    plt.tight_layout()
    plt.show()
else:
    print("No numerical columns found in the dataset")

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    plt.figure(figsize=(10, 4))
    df[col].value_counts().plot(kind='bar', color='steelblue')
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Correlation Analysis

In [ ]:
numerical_cols = df.select_dtypes(include=['int64','float64']).columns
if len(numerical_cols) > 1:
    corr_matrix = df[numerical_cols].corr()
    plt.figure(figsize=(12, 8))
    sns.heatmap(corr_matrix, 
                annot=True, 
                fmt='.2f', 
                cmap='coolwarm',
                center=0,
                square=True)
    plt.title('Correlation Matrix')
    plt.tight_layout()
    plt.show()
    
    print("\n=== Top Correlated Pairs ===")
    corr_pairs = corr_matrix.unstack()
    corr_pairs = corr_pairs[corr_pairs != 1.0]
    corr_pairs = corr_pairs.abs().sort_values(ascending=False)
    print(corr_pairs.head(10))
else:
    print("Not enough numerical columns for correlation analysis")

## 7. Outlier Detection

In [ ]:
numerical_cols = df.select_dtypes(include=['int64','float64']).columns
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5 * IQR) | 
                  (df[col] > Q3 + 1.5 * IQR)]
    print(f"{col}: {len(outliers)} outliers detected")

if len(numerical_cols) > 0:
    plt.figure(figsize=(15, 8))
    df[numerical_cols].boxplot()
    plt.title('Boxplots — Outlier Detection')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No numerical columns available for boxplot visualization")

## 8. Duplicate Detection

In [ ]:
duplicates = df.duplicated().sum()
print(f"Total duplicate rows: {duplicates}")
print(f"Percentage: {(duplicates/len(df)*100).round(2)}%")

if duplicates > 0:
    print("\n=== Sample Duplicate Rows ===")
    display(df[df.duplicated()].head())

## 9. Scatterplots — Top Correlated Pairs

In [ ]:
numerical_cols = df.select_dtypes(include=['int64','float64']).columns
if len(numerical_cols) > 1:
    corr_matrix = df[numerical_cols].corr()
    corr_pairs = corr_matrix.unstack()
    corr_pairs = corr_pairs[corr_pairs != 1.0]
    corr_pairs = corr_pairs.abs().sort_values(ascending=False)
    top_pairs = corr_pairs.head(2).index.tolist()
    
    for col1, col2 in top_pairs:
        plt.figure(figsize=(8, 5))
        sns.scatterplot(data=df, x=col1, y=col2, alpha=0.5)
        plt.title(f'{col1} vs {col2}')
        plt.tight_layout()
        plt.show()
else:
    print("Not enough numerical columns for scatterplot analysis")

## 10. Executive Summary

This dataset comprises 45,213 banking marketing campaign records with 19 columns capturing customer demographics, account characteristics, and campaign interaction history. However, the dataset suffers from a critical structural deficiency: the header row appears corrupted, with only the first column meaningfully named ('banking marketing') and 18 remaining columns labeled as generic 'Unnamed' fields. All columns are stored as object (text) data types despite containing numeric and categorical information, indicating improper data ingestion. While the underlying banking marketing data appears substantive with low overall missing rates (0.04-0.11%), the data structure requires immediate remediation before any reliable analytical work can proceed.

## 11. Key Findings

- **Critical Header Issue**: The first column header 'banking marketing' does not match the attribute description which refers to 'Customer id and age'. This indicates a fundamental import error affecting all downstream analysis.

- **Data Type Misalignment**: All 19 columns are classified as 'object' type despite containing numeric values (ages, salaries, balances, durations). The numeric columns show statistical summaries with unique counts and frequency distributions, confirming they are incorrectly typed as strings.

- **Low Overall Missing Rates**: Most columns have minimal missing data (0.02-0.11% missing). The exceptions are Unnamed: 1 (21 missing, 0.05%), Unnamed: 12 (50 missing, 0.11%), and Unnamed: 18 (30 missing, 0.07%), which are relatively negligible.

- **High Cardinality in Numeric Columns**: Unnamed: 1 and Unnamed: 3 show 45,192 and 45,212 unique values respectively out of 45,213 records, confirming these are continuous numeric variables (likely age and salary or balance). Several other columns show extremely high uniqueness (Unnamed: 13 with 2,648 unique values), consistent with numeric identifiers or continuous measures.

- **Skewed Categorical Distributions**: Unnamed: 10 (contact method) shows 29,285 occurrences of 'cellular' (64.8%) versus 'telephone', indicating strong preference for mobile contact channels. Unnamed: 8 (likely housing loan) shows 44,396 'no' responses (98.2%), revealing a heavily imbalanced outcome distribution.

## 12. Data Quality Assessment

**Missing Data Impact:**
Missing values are sparse across the dataset:
- Unnamed: 1 (presumed age or demographic): 21 missing (0.05%)
- Unnamed: 12 (likely previous outcome): 50 missing (0.11%)
- Unnamed: 18 (likely response/target): 30 missing (0.07%)
- All other columns: 1 missing value each (0.002%)

The missing values are negligible in volume and localized, presenting minimal analytical risk if handled appropriately through deletion or imputation.

**Outlier Severity:**
Without properly typed data, formal outlier detection cannot be reliably performed. However, the frequency distributions suggest:
- Unnamed: 8 ('no' at 98.2%) and Unnamed: 6 ('yes' at 82% based on freq) indicate extreme class imbalance typical in marketing response datasets
- High unique value counts in Unnamed: 1 and Unnamed: 3 suggest numeric ranges that likely contain valid outliers in age (18-95 range) and salary/balance distributions

**Duplicate Records Assessment:**
The first column 'banking marketing' shows 45,213 unique values out of 45,213 records, suggesting no exact duplicates. However, duplicate detection on business logic (same customer contacted multiple times) cannot be assessed until column headers are corrected.

**Overall Data Quality Rating: POOR**

Justification: While the underlying data appears substantive with minimal missing values and no apparent duplicates, the critical failure is structural integrity. The corrupted header row and universal object-type misclassification render the dataset non-functional for analysis without significant preprocessing. The data quality issue is not statistical but infrastructural—this dataset cannot be reliably analyzed in its current state.

## 13. Business Implications

**Current Analytical Limitations:**
In its present state, this dataset cannot support business decision-making. The corrupted column headers mean we cannot reliably map the 18 'Unnamed' columns to their intended business attributes, creating fundamental ambiguity about what is actually being measured.

**Inferred Value (if corrected):**
Assuming the attribute descriptions are accurate and correctly mappable, this dataset contains highly valuable marketing intelligence:

- **Customer Segmentation Opportunity**: The jobedu field (job + education combined) combined with age, salary, and balance enables rich demographic and socioeconomic segmentation. The 67 unique job-education combinations suggest detailed occupational stratification.

- **Campaign Effectiveness Analysis**: The presence of contact method (cellular vs telephone), day, month, duration, campaign frequency, and previous campaign outcomes creates a complete funnel for measuring marketing channel efficiency. The heavy skew toward 'cellular' (64.8%) suggests this is the primary engagement channel.

- **Predictive Power in Previous Behavior**: The pdays and previous columns indicate prior campaign history, with poutcome tracking success/failure/unknown. This creates the foundation for predictive modeling—customers with prior successful outcomes likely have higher conversion propensity.

- **Most Analytically Valuable Columns (if properly typed)**: age, salary, balance, marital status, duration, previous campaign outcomes, and contact method. These enable cohort analysis, RFM-style segmentation, and response likelihood prediction.

- **Key Segments to Investigate**: Once corrected, immediate investigation should target (1) customers with prior 'success' outcomes to identify characteristics of high-propensity prospects, (2) housing loan holders vs non-holders as a proxy for financial health, and (3) optimal contact timing by analyzing conversion rates across months and days.

## 14. Risk Flags

**CRITICAL: Data Integrity Failure**
The corrupted header row is the primary risk. The attribute descriptions reference 'Customer id', 'age', 'salary', etc., but these names do not exist in the actual column headers. This creates severe mapping risk—without manual verification, we cannot confirm that Unnamed: 1 is actually 'age' or if columns are shifted. Any analysis proceeding without header verification will produce unreliable results.

**Type Conversion Failure**
All columns being stored as 'object' (text) despite containing numeric and categorical data indicates either:
- Malformed import (mixed data types in columns, common with poorly standardized entry)
- Encoding issues causing numeric values to be read as strings
- Source data quality problems in the original system

This creates risk of silent analysis failures where numeric operations fail or produce wrong results on string data.

**Missing Data Concentration Risk**
While overall missing rates are low, Unnamed: 1 shows 21 missing values while maintaining 45,192 unique values—statistically impossible unless the missing pattern is systematic. This suggests potential data collection process failure for this specific field.

**Extreme Class Imbalance Risk**
Unnamed: 8 shows 98.2% 'no' responses (likely housing loan), and Unnamed: 6 shows 82% 'yes'. If these are predictive features, severe class imbalance will bias any machine learning models. If Unnamed: 18 (inferred as response variable) is similarly imbalanced, this is expected in marketing campaigns but requires specific handling (resampling, class weights).

**Potential Data Collection Issues**
The 'unknown' category appearing frequently in Unnamed: 10 (contact method, 30,277 records = 67% frequency for unknown values) suggests incomplete data capture. This indicates historical records with poor documentation or data entry shortcuts.

**Bias Risks**
- Geographic bias: No location data visible, but if campaigns focused on specific regions, results won't generalize
- Temporal bias: Campaign dates are present (day, month) but no year information—if this spans multiple years with economic changes, trend analysis will be confounded
- Selection bias: Unknown sampling mechanism—are these all customers contacted, or a filtered subset?

## 15. Recommendations

**PHASE 1: CRITICAL DATA RECONSTRUCTION (MUST DO FIRST)**
1. Re-import the source file, carefully preserving the original header row. Use pandas with explicit header=0 parameter and do NOT allow auto-naming of columns.
2. Manually verify the mapping between the 19 columns in the file and the 19 attributes described in the attribute dictionary. Document any discrepancies.
3. Save this corrected structure as a validated baseline before proceeding.

**PHASE 2: DATA TYPE CONVERSION**
Convert from object to appropriate types:
- Columns mapping to 'Customer id': keep as string or convert to integer if purely numeric
- 'age': convert to int64
- 'salary', 'balance': convert to float64 (may contain decimals)
- 'day', 'duration', 'campaign', 'pdays', 'previous': convert to int64
- 'marital', 'jobedu', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'response': convert to category or string depending on use case

**PHASE 3: COLUMNS TO DROP**
- **'banking marketing' (Unnamed: 0)**: If this is 'Customer id', drop it before modeling. It provides no predictive value and creates data leakage risk if models are tuned per-customer. Keep only for record tracking if needed separately.

**PHASE 4: COLUMNS TO SPLIT**
- **'jobedu'**: The attribute description explicitly states this combines job AND education information. This combined feature should be split into two separate columns:
  - Extract job category (management, technician, services, etc.) into 'job'
  - Extract education level (primary, secondary, tertiary, unknown) into 'education'
  - These separate features will have higher predictive power and interpretability in segmentation.

**PHASE 5: MISSING DATA HANDLING**
- For Unnamed: 1 (21 missing): Investigate if missing patterns are random. If <1%, delete rows. If systematic, impute with median/mode after determining column identity.
- For Unnamed: 12 (50 missing) and Unnamed: 18 (30 missing): Critical if these are 'poutcome' and 'response' (target variable). If response is the target, delete rows with missing response. If poutcome (previous campaign outcome), impute with 'unknown' category.
- For scattered single-value missing cells: Safe to delete or impute with category mode.

**PHASE 6: RECOMMENDED SCATTERPLOTS**
1. **X: age (Unnamed: 2), Y: balance (Unnamed: 4), Color: response (Unnamed: 18)**
   - Expected insight: Age cohorts and financial capacity are primary drivers of term deposit adoption. This reveals whether older/wealthier customers are more receptive, guiding segment-specific campaign strategies.
   
2. **X: duration (Unnamed: 13), Y: response (Unnamed: 18), Color: contact (Unnamed: 10)**
   - Expected insight: Call duration is a strong proxy for engagement quality. This shows whether cellular vs telephone contact methods differ in their relationship between talk time and conversion, revealing channel-specific optimization opportunities.

**PHASE 7: FURTHER ANALYSIS**
- **Cohort Analysis**: Segment customers by job-education combination and analyze response rates by cohort. Identify which professional groups respond best to minimize wasted outreach.
- **Time Series Analysis**: Analyze campaign performance across days of month and months (seasonality). Identify optimal contact windows.
- **RFM-Style Analysis**: Use pdays, previous, and campaign frequency to create a recency-frequency scoring model predicting response likelihood.
- **Propensity Model**: Build logistic regression or gradient boosting model using duration, contact method, previous outcomes, and demographics to score prospects and optimize targeting.
- **Channel Comparison**: Analyze conversion rates, average call duration, and cost-per-conversion by contact method (cellular vs telephone) to optimize channel allocation.

## 16. Limitations

**Cannot Be Concluded From This Dataset Alone:**
- **Causality**: We can identify correlations between customer characteristics and term deposit adoption, but cannot determine causation. For example, if older customers are more likely to open term deposits, we cannot conclude that age causes deposit behavior—confounding variables (income stability, life stage, risk aversion) drive both.
- **Generalization**: Without knowledge of geographic scope, time period, and sampling methodology, results may not generalize beyond this specific campaign or institution.
- **Customer Lifetime Value**: The dataset captures only term deposit products. Real marketing ROI requires cost data (contact costs, customer lifetime value of deposits) absent here.
- **Competitive Context**: No competitive or market context exists. High response rates might reflect market conditions rather than campaign quality.

**Assumptions Made During This Analysis:**
1. **Column Mapping Assumption**: I have assumed (without verification) that the 19 columns match the 19 attribute descriptions in order. If columns are shifted or missing, all interpretations are invalid.
2. **Object Storage Assumption**: I have assumed that object-type columns contain numeric data that was mislabeled during import, not that they legitimately contain text-only data.
3. **Data Completeness Assumption**: I have assumed that missing values are truly missing (not encoded as special values like 'NA' or '0' within the text).
4. **Recency Assumption**: I have assumed this is recent campaign data relevant for current strategy, though no date range is provided.
5. **Single Institution Assumption**: I have assumed all records come from a single bank or institution with consistent practices.

**Sample Size and Time Period Concerns:**
- **Sample Size**: 45,213 records is substantial for statistical inference but modest for detecting small effect sizes across multiple segments. If broken down by job-education (67 combinations) and month (12 months), individual segments may contain <600 observations, limiting precision.
- **Time Period Unknown**: The month and day columns indicate when contacts occurred, but the year(s) covered are unknown. Multi-year data would reflect economic cycles; single-month data would reflect seasonal effects. This severely limits trend analysis without clarification.
- **Temporal Bias**: Banking products were significantly restructured post-2008 financial crisis and post-2020 pandemic. If this dataset spans these periods, results will be confounded by structural economic changes.

**Data Collection Methodology Unknown:**
- Are these all customers contacted (census) or a sample? If sampled, what's the sampling method? Bias risk is high.
- Response mechanism unknown: Did customers respond immediately, or was follow-up required? This affects causality interpretation.
- Measurement validity unknown: Are conversions self-reported or verified through transaction records?